# Explainable AI (XAI) with SHAP

This notebook uses SHAP (SHapley Additive exPlanations) to explain individual predictions and evaluate global feature importance. This provides transparency to clinical decisions made by the machine learning model.

In [ ]:
import os
import sys
import pandas as pd
import shap
import matplotlib.pyplot as plt

# Append parent directory to path so we can import our modules
sys.path.append(os.path.abspath(os.path.join("..")))

## 1. Local Explanation for Patient Predictions

We use our `MaatriSHAPExplainer` to compute local SHAP feature contributions for a sample High-Risk case.

In [ ]:
from ml.explainability.shap_explainer import MaatriSHAPExplainer
from ml.risk_prediction.predict import predict_risk

explainer = MaatriSHAPExplainer(models_dir=os.path.join("..", "models"))

# Define patient (Elevated BP, High Blood Sugar, age 36)
patient_data = {
    'Age': 36,
    'SystolicBP': 140,
    'DiastolicBP': 95,
    'BS': 8.5,
    'BodyTemp': 98.6,
    'HeartRate': 85
}

# Get prediction first
pred_result = predict_risk(patient_data, models_dir=os.path.join("..", "models"))
risk_label = pred_result['risk_category']
risk_mapping = {'Low Risk': 0, 'Mid Risk': 1, 'High Risk': 2}
pred_idx = risk_mapping[risk_label]

print(f"Prediction: {risk_label} ({pred_result['confidence_score'] * 100:.2f}% confidence)")

# Compute SHAP for the predicted class
explanation = explainer.explain_instance(patient_data, pred_idx)
print("\n=== Feature SHAP Contributions ===")
for feat, val in explanation['feature_contributions'].items():
    print(f"{feat}: {val:+.4f}")

## 2. Generate Local Explanation Plot

We generate a horizontal bar chart showing how feature contributions push the prediction relative to the base value.

In [ ]:
fig = explainer.generate_explanation_plot(explanation)
plt.show()

## 3. Global Explainability Summary

Let's visualize SHAP values globally using a subset of test data.

In [ ]:
import joblib
from ml.risk_prediction.preprocessing import preprocess_train_data

# Preprocess test dataset to get an evaluation set
csv_path = os.path.join("..", "datasets", "maternal_health_risk.csv")
X_train, X_test, y_train, y_test, scaler, _ = preprocess_train_data(
    csv_path, test_size=0.2, random_state=42
)

# Compute SHAP values for test set
shap_values_test = explainer.explainer.shap_values(X_test)

# Plot summary plot (for high risk class: index 2)
plt.figure(figsize=(10, 6))
if isinstance(shap_values_test, list):
    # Random Forest
    shap.summary_plot(shap_values_test[2], X_test, show=False)
else:
    # XGBoost 3D array
    shap.summary_plot(shap_values_test[:, :, 2], X_test, show=False)
plt.title("Global Feature Influence for predicting 'High Risk'", fontsize=14, pad=15)
plt.show()